# 02 — Fine-tuning EdgeTAM on 512×512 thermal

Takes the mask store from notebook 01 and produces
`checkpoints/edgetam_thermal_512.pt`, which drops into the existing configs,
exporter and engine build **with nothing else changed** — the fine-tune moves
weights, never architecture.

> **Use a fresh runtime.** This installs EdgeTAM, which claims the package name
> `sam2`. Notebook 01's SAM 2.1 teacher cannot coexist with it.

---

## The decision: partial fine-tuning, not LoRA

| | partial fine-tune | LoRA |
|---|---|---|
| **memory** | EdgeTAM is **13.9 M parameters in total** (4.92 encoder / 2.96 memory attention / 1.62 memory encoder / 4.41 head). Nothing about training it is memory-bound on an A100, or even an L4 | solves a problem we do not have |
| **deployment** | weights change, the ONNX graph is identical | a merged adapter *is* just weights — it buys nothing downstream; unmerged, it adds ops to the hot path |
| **coverage** | reaches everything | targets `nn.Linear`. The domain shift here is in the **convolutional RepViT trunk**, which is LoRA's worst case |
| **QAT afterwards** | the same loop, with `mtq.quantize` applied | needs real weight updates anyway, so the adapter would be unwound |

LoRA's one genuine benefit — regularisation on a small single-class dataset —
is bought more cheaply and far more precisely by **freezing the right modules**.

### What stays frozen, and why

**The whole memory path**: `memory_attention`, `memory_encoder`,
`spatial_perceiver`, and the learned memory tokens. Three independent reasons:

1. **It is the write port of a recurrent loop.** This project already measured
   what a systematic change there costs: quantising the memory encoder alone
   gave mean IoU 0.9397, and — unlike every other module — the damage was a
   *sustained decline across the clip* rather than isolated dropouts, because
   its output is what the bank stores and the next seven frames read back
   (`docs/tensorrt_fp16.md`).
2. **Its ONNX rewrite is the intricate one** — fixed memory slots, tiled RoPE
   tables, an additive attention mask. Retraining the weights those graphs were
   derived from invites a silent mismatch between checkpoint and engine.
3. **It operates on abstract features, not pixels.** Thermal-versus-RGB is an
   encoder problem.

### Two more choices worth knowing about

- **The model trains in `eval()` mode.** RepViT is full of batch norms, and
  letting their statistics drift would break the match with the engines
  TensorRT folds them into, and invalidate any INT8 calibration taken before
  it. Independently, SAM 2 withholds `object_score_logits` in training mode.
- **No teacher forcing.** Every frame conditions on the memory *the model
  itself* wrote. Feeding ground truth into the bank would train a model that
  has never seen its own mistakes — which is precisely the failure being fixed.

In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/content/sam-dedection")
if not REPO.exists():
    !git clone -q https://github.com/yigitkayabagci/sam-dedection.git {REPO}
os.chdir(REPO)
sys.path.insert(0, str(REPO))

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!bash scripts/setup_edgetam.sh 2>&1 | tail -5
!pip install -q -r requirements.txt

In [ ]:
# The loop's contract with EdgeTAM is pinned by tests that need no GPU and no
# checkpoint. If these fail, nothing below is worth running.
!python -m unittest tests.test_clip_loop tests.test_training_losses \
    tests.test_antiuav_dataset tests.test_accuracy 2>&1 | tail -3

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/anti-uav410")
WORK = Path("/content/drive/MyDrive/edgetam-thermal")   # written by notebook 01
CKPT = REPO / "checkpoints"; CKPT.mkdir(exist_ok=True)

import json
manifest = json.loads((WORK / "manifest.json").read_text())
SIZE = manifest["model_input"]
FRAME_W, FRAME_H = manifest["frame_size"]
CLIP_LEN, CLIP_STRIDE = manifest["clip"]["length"], manifest["clip"]["stride"]
print(json.dumps(manifest["acceptance"], indent=2))

In [ ]:
# --- Clips, paired with their pseudo-masks -----------------------------
from src.training import list_sequences, load_masks, sample_clips

def build(split, jitter):
    sequences = [s for s in list_sequences(DATA, split)
                 if s.name in set(manifest["sequences"][split])]
    stores = {s.name: load_masks(WORK / "labels" / split / s.name / "pseudo_masks.npz")[1]
              for s in sequences}
    clips = sample_clips(sequences, length=CLIP_LEN, stride=CLIP_STRIDE, size=SIZE,
                         frame_size=(FRAME_W, FRAME_H), jitter=jitter, seed=0)
    return clips, stores

train_clips, train_masks = build("train", jitter=32)
val_clips, val_masks = build("val", jitter=0)
print(f"train {len(train_clips)} clips / val {len(val_clips)} clips "
      f"of {CLIP_LEN} frames")

In [ ]:
# --- The model ---------------------------------------------------------
# image_size_overrides also fixes the cross-attention rotary table, which does
# not self-adjust to a new resolution (src/trackers/_hydra_overrides.py).
import torch
from sam2.build_sam import build_sam2_video_predictor
from src.trackers._hydra_overrides import image_size_overrides

model = build_sam2_video_predictor(
    "configs/edgetam.yaml",
    "third_party/EdgeTAM/checkpoints/edgetam.pt",
    device="cuda",
    hydra_overrides_extra=image_size_overrides(SIZE),
)
model.eval()   # deliberate -- see the header
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters, "
      f"image_size={model.image_size}")

In [ ]:
from src.training.finetune import Rates, apply_freeze, param_groups, summarise_freeze

counts = apply_freeze(model, "head")
print(summarise_freeze(counts, model))

# Assert the policy rather than trusting the print: a run that quietly trained
# the memory path would only show up as a bad checkpoint hours later.
for name, p in model.named_parameters():
    if name.startswith(("memory_attention", "memory_encoder", "spatial_perceiver")):
        assert not p.requires_grad, name

## First: overfit one clip

Before spending GPU hours, prove the loop can learn *anything*. One clip, no
augmentation, frame 0 included in the loss — this should collapse towards zero
within a couple of hundred steps.

If it does not, the problem is in the plumbing (prompt coordinates, mask
alignment, the memory bookkeeping), and no amount of real training will fix it.
This is the cheapest possible place to find that out.

In [ ]:
import copy
from src.training.clip_loop import clip_losses, collate

probe_clip = next(c for c in train_clips
                  if len(train_masks[c.sequence.name]) > CLIP_LEN)
probe = collate([probe_clip], [train_masks[probe_clip.sequence.name]], "cuda")

snapshot = copy.deepcopy(model.state_dict())
opt = torch.optim.AdamW(param_groups(model, Rates(head=3e-4)))
history = []
for step in range(120):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss, terms = clip_losses(model, probe, skip_first=False)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
    opt.step()
    history.append(float(loss))
    if step % 20 == 0:
        print(f"step {step:>4}  loss {float(loss):7.4f}   " +
              "  ".join(f"{k} {v:.3f}" for k, v in terms.items()))

drop = history[0] / max(min(history[-10:]), 1e-6)
print(f"\nloss fell {drop:.1f}x  ({history[0]:.3f} -> {min(history[-10:]):.3f})")
assert drop > 3.0, ("the loop cannot even overfit one clip. Check prompt "
                    "coordinates, mask alignment and the memory bookkeeping "
                    "before training on anything larger.")
model.load_state_dict(snapshot)   # throw the overfit away

## Training

Two stages. The head first, alone, so the mask decoder and the object-score
head adapt to thermal statistics before the features under them start moving;
then the encoder joins at a tenth of the rate.

The batch dimension is **clips**, not frames. SAM 2 batches *objects*, and each
row of that batch carries its own memory, its own object pointer and its own
object score — the attention never mixes rows. So N independent clips ride the
same machinery N tracked objects would.

Memory at 512², bf16: roughly 4 GB per clip in the batch. `BATCH = 4` fits an
A100 40 GB comfortably; drop to 1 on an L4 and raise `ACCUM` to keep the
effective batch.

In [ ]:
import numpy as np
from src.training.antiuav import iter_clips
from src.training.finetune import EMA, save_checkpoint
from tqdm.auto import tqdm

BATCH, ACCUM = 4, 1
STAGES = [("head", 2, Rates(head=1e-4)),
          ("encoder", 4, Rates(head=5e-5, neck=5e-5, trunk=1e-5))]

def batches(clips, stores, size, seed):
    pool = list(iter_clips(clips, seed))
    for i in range(0, len(pool) - size + 1, size):
        chunk = pool[i:i + size]
        yield collate(chunk, [stores[c.sequence.name] for c in chunk], "cuda")

@torch.no_grad()
def validate(limit=40):
    total = [float(clip_losses(model, b)[0])
             for b in batches(val_clips, val_masks, BATCH, seed=1)][:limit]
    return float(np.mean(total)) if total else float("nan")

ema = EMA(model, decay=0.999)
best, log = float("inf"), []

for stage, epochs, rates in STAGES:
    print(f"\n===== stage {stage!r}: {epochs} epoch(s) =====")
    print(summarise_freeze(apply_freeze(model, stage), model))
    opt = torch.optim.AdamW(param_groups(model, rates))
    steps = epochs * (len(train_clips) // BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in opt.param_groups], total_steps=max(steps, 1),
        pct_start=0.1)

    for epoch in range(epochs):
        bar = tqdm(batches(train_clips, train_masks, BATCH, seed=epoch),
                   total=len(train_clips) // BATCH, desc=f"{stage} e{epoch}")
        for step, batch in enumerate(bar):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss, terms = clip_losses(model, batch)
            (loss / ACCUM).backward()
            if (step + 1) % ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for g in opt.param_groups for p in g["params"]], 1.0)
                opt.step(); opt.zero_grad(set_to_none=True); sched.step()
                ema.update(model)
            bar.set_postfix(loss=f"{float(loss):.3f}",
                            **{k: f"{v:.2f}" for k, v in terms.items()})

        with ema.applied(model):
            score = validate()
            if score < best:
                best = score
                save_checkpoint(model, CKPT / "edgetam_thermal_512.pt",
                                {"stage": stage, "epoch": epoch, "val_loss": score,
                                 "image_size": SIZE, "dataset": "Anti-UAV410"})
                marker = "  <- saved"
            else:
                marker = ""
        log.append({"stage": stage, "epoch": epoch, "val": score})
        print(f"  epoch {epoch}: val clip loss {score:.4f}{marker}")

## Did it actually help?

Clip loss is a proxy. The number that decides anything is **state accuracy on
held-out sequences, measured through the deployment path** — the same
`VideoTracker` the Orin runs, prompted once and left to propagate.

`tools/eval_antiuav.py` also reports **dropout episodes**: how often a visible
target was lost and for how long. That is the statistic the mean hides, and it
is the one that describes this project's actual failure — one hard frame writes
`no_obj_ptr` into the memory bank and the next frames read it back. If the
`exist` supervision worked, the episodes get *shorter*, not just rarer.

In [ ]:
# Before: the stock checkpoint at 512. After: the fine-tuned one. Same
# resolution, same tracker, same sequences -- only the weights differ.
!python tools/eval_antiuav.py --data {DATA} --split val --limit 12 \
    --tracker edgetam --config configs/edgetam_512.yaml --mode crop \
    --json /content/eval_stock.json 2>&1 | tail -20

In [ ]:
!python tools/eval_antiuav.py --data {DATA} --split val --limit 12 \
    --tracker edgetam --config configs/edgetam_512_thermal.yaml --mode crop \
    --json /content/eval_thermal.json 2>&1 | tail -20

In [ ]:
import json
import matplotlib.pyplot as plt

stock = json.loads(Path("/content/eval_stock.json").read_text())["sequences"]
tuned = json.loads(Path("/content/eval_thermal.json").read_text())["sequences"]
by_name = {s["name"]: s for s in tuned}

def weighted(rows, key):
    frames = sum(r["frames"] for r in rows)
    return sum(r[key] * r["frames"] for r in rows) / max(frames, 1)

print(f"{'':<12}{'state acc':>12}{'success AUC':>14}{'lost frames':>14}")
for label, rows in (("stock", stock), ("fine-tuned", tuned)):
    lost = sum(sum(r["dropout_lengths"]) for r in rows)
    print(f"{label:<12}{weighted(rows, 'state_accuracy'):>12.4f}"
          f"{weighted(rows, 'success_auc'):>14.4f}{lost:>14}")

fig, ax = plt.subplots(figsize=(7, 4))
names = [s["name"] for s in stock]
ax.barh(range(len(names)), [by_name[n]["state_accuracy"] - s["state_accuracy"]
                            for n, s in zip(names, stock)])
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("state accuracy: fine-tuned - stock")
plt.tight_layout(); plt.show()

In [ ]:
# --- Keep it -----------------------------------------------------------
import shutil
shutil.copy(CKPT / "edgetam_thermal_512.pt", WORK / "edgetam_thermal_512.pt")
(WORK / "finetune_log.json").write_text(json.dumps(
    {"stages": [s[0] for s in STAGES], "batch": BATCH, "log": log,
     "best_val": best}, indent=2) + "\n")
print(f"checkpoint -> {WORK / 'edgetam_thermal_512.pt'}")

## What you have, and what to check

`checkpoints/edgetam_thermal_512.pt` — in EdgeTAM's own `{"model": state_dict}`
layout, so `configs/edgetam_512_thermal.yaml` already points at it and the
export/build chain needs no change:

```bash
python tools/export_edgetam_onnx.py --outdir models512_ft/ --image-size 512 \
    --checkpoint checkpoints/edgetam_thermal_512.pt --verify
python tools/build_trt_engines.py --outdir models512_ft/ --max-batch 4
```

**Read the comparison honestly.** Three outcomes and what each one means:

| what you see | what it means |
|---|---|
| state accuracy up, dropout episodes *shorter* | the `exist` supervision landed — this is the win being sought |
| state accuracy up, dropouts unchanged | masks got better, the object head did not. Raise `Weights.object_score` |
| state accuracy down on some sequences | check whether those are the `full512` clips: a target that outruns a fixed window trains on a distorted resize. `manifest["clip"]` and `Clip.native` tell you |

**Next:** `03_quantization_v2_int8.ipynb`. It calibrates on *this* checkpoint —
INT8 scales taken from the stock model would be calibrated for the wrong
activation distributions.